[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/06-agentic-ai/04-mcp_server_advanced.ipynb)

In [1]:
# !pip install mbox mcp

# MCP Server, Advanced: One Index, Many Endpoints

`03` built an MCP server with a single tool, and rebuilt its index from a CSV every time the server started. Both of those choices were fine for a first example and stop being fine the moment a server restarts often, or needs to answer more than one kind of question. This notebook covers two things that matter for a real deployment: building the index once and loading the compiled result instead of rebuilding it, and exposing several distinct search views, not just one tool, from that single loaded index.

In this notebook you will:

1. Build an M|BOX index once and save it to a single file with `to_binary()`
2. Write a server that loads that file with `load_binary()` instead of rebuilding from source data
3. Expose four different search views, fuzzy name, fuzzy name plus description, description only, and an exact id lookup, all against that one loaded index
4. Call all four for real, including a genuine multi-field search where two fields both move the score

> Note: this notebook spawns `mcp_server_advanced.py` as a separate process and talks to it over stdio. Run the cells in order.

In [2]:
import pandas as pd

## 1. Build the index once

A server that rebuilds its index from a CSV on every single startup pays that cost, however small, every time it restarts, and has to keep the raw data file around just to boot. `TableIndex.to_binary()` compiles the index once and writes it to a single file; `TableIndex.load_binary()` loads that compiled file back in milliseconds, no source DataFrame required. This is the same one-time build, many-time reuse pattern as `01-getting-started/03_saving_and_loading_indexes.ipynb`, applied here to a server process instead of a notebook kernel.

In [3]:
from mbox.indexing import TableIndexer

catalog = pd.read_csv("datasets/product_catalog.csv")
index = TableIndexer.create_index(
    catalog, index_columns=["product_name", "description", "unit_price", "product_id"], tmp_dir="tmp_index_build"
)
index.to_binary("product_catalog_index.zip")
print("Saved product_catalog_index.zip")

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object
Saved product_catalog_index.zip


## 2. The server: four views over one loaded index

`%%writefile` below writes out `mcp_server_advanced.py`. It loads `product_catalog_index.zip` once at startup, via `TableIndex.load_binary()`, and every tool it exposes runs a different `TableRecallConfig` against that same loaded index, not a separate index per tool:

- **`search_products`** - fuzzy name match, optionally constrained by a maximum price, the same tool as `03`
- **`search_products_multi_field`** - fuzzy name *and* a rough description, both weighted and scored, for when the user describes what they want as much as names it
- **`search_by_description`** - fuzzy match on the description alone, for when the user doesn't remember the name at all
- **`get_product_by_id`** - an exact, no-fuzziness lookup by `product_id`, for when something upstream has already resolved an id and just needs the full record

The stdout redirect around the import is the same technique `03` used, and the same known M|BOX issue applies, see the note in the next section.

In [4]:
%%writefile mcp_server_advanced.py
"""An MCP server exposing several M|BOX-backed search views over one pre-built index.

MCP over stdio uses stdout exclusively for JSON-RPC protocol messages, so anything
else written to it corrupts the transport. M|BOX itself logs a license line and a
startup banner as soon as it's imported, so we redirect stdout to the null device
for the import to keep those off the wire. This catches the synchronous log line
reliably; see the notebook for a caveat on the rest.
"""
import os
import time

_stdout_fd = os.dup(1)
_devnull_fd = os.open(os.devnull, os.O_WRONLY)
os.dup2(_devnull_fd, 1)
try:
    import pandas as pd
    from mbox.indexing import TableIndex
    from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

    time.sleep(2)
finally:
    os.dup2(_stdout_fd, 1)
    os.close(_stdout_fd)
    os.close(_devnull_fd)

from mcp.server import MCPServer

# Load the index built once in the notebook instead of rebuilding it from CSV on
# every server start. Every tool below is a different search view over this one
# loaded index, not a separate index.
index = TableIndex.load_binary("product_catalog_index.zip")
catalog = pd.read_csv("datasets/product_catalog.csv")

server = MCPServer("mbox-catalog-advanced")


def _text_only_match(product_name: str, max_results: int) -> pd.DataFrame:
    config = TableRecallConfig(
        fields=[TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                        minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
        max_results=max_results, min_total_match_value=0, include_field_scores=True
    )
    return index.match(queries=pd.DataFrame({"product_name": [product_name]}), config=config)


@server.tool()
def search_products(product_name: str, max_price: float | None = None, max_results: int = 3) -> dict:
    """Search the product catalog by name, tolerating typos, optionally constrained by a maximum price."""
    if max_price is None:
        r = _text_only_match(product_name, max_results)
        if len(r) == 0 or r["index_row"].iloc[0] == -1:
            return {"found": False, "matches": []}
        matches = [{"product_name": row["product_name_candidate"],
                    "match_confidence": int(row["product_name_score"]),
                    "overall_score": int(row["overall_score"])} for _, row in r.iterrows()]
        return {"found": True, "matches": matches}

    fields = [
        TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                minimum_quality=0, weight=70, mode=TableRecallMode.APPROX),
        TableRecallFieldConfig(input_column="unit_price", indexed_column="unit_price",
                                minimum_quality=0, weight=30, mode=TableRecallMode.NUM_LOWER)
    ]
    config = TableRecallConfig(fields=fields, max_results=max_results, min_total_match_value=0, include_field_scores=True)
    r = index.match(queries=pd.DataFrame({"product_name": [product_name], "unit_price": [max_price]}), config=config)

    if len(r) > 0 and r["index_row"].iloc[0] != -1:
        matches = [{"product_name": row["product_name_candidate"],
                    "match_confidence": int(row["product_name_score"]),
                    "overall_score": int(row["overall_score"])} for _, row in r.iterrows()]
        return {"found": True, "matches": matches}

    unconstrained = _text_only_match(product_name, max_results=1)
    if len(unconstrained) > 0 and unconstrained["index_row"].iloc[0] != -1:
        row = unconstrained.iloc[0]
        actual_price = catalog.loc[catalog["product_name"] == row["product_name_candidate"], "unit_price"].iloc[0]
        return {
            "found": False,
            "matches": [],
            "note": "A matching product exists but exceeds the requested max_price.",
            "closest_match": {
                "product_name": row["product_name_candidate"],
                "match_confidence": int(row["product_name_score"]),
                "actual_price": float(actual_price)
            }
        }

    return {"found": False, "matches": []}


@server.tool()
def search_products_multi_field(product_name: str, description: str | None = None, max_results: int = 5) -> dict:
    """Search by product name and, optionally, a rough description of what the product does.
    Both fields are weighted and scored independently. Use this when the user describes what
    they're looking for as much as, or instead of, naming it precisely."""
    if description is None:
        r = _text_only_match(product_name, max_results)
    else:
        fields = [
            TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                    minimum_quality=0, weight=50, mode=TableRecallMode.APPROX),
            TableRecallFieldConfig(input_column="description", indexed_column="description",
                                    minimum_quality=0, weight=50, mode=TableRecallMode.APPROX),
        ]
        config = TableRecallConfig(fields=fields, max_results=max_results, min_total_match_value=0, include_field_scores=True)
        r = index.match(queries=pd.DataFrame({"product_name": [product_name], "description": [description]}), config=config)

    if len(r) == 0 or r["index_row"].iloc[0] == -1:
        return {"found": False, "matches": []}

    matches = []
    for _, row in r.iterrows():
        match = {
            "product_name": row["product_name_candidate"],
            "overall_score": int(row["overall_score"]),
            "field_scores": {"product_name": int(row["product_name_score"])},
        }
        if description is not None:
            match["field_scores"]["description"] = int(row["description_score"])
        matches.append(match)
    return {"found": True, "matches": matches}


@server.tool()
def search_by_description(description: str, max_results: int = 3) -> dict:
    """Search the product catalog by what a product does, for when the user doesn't remember its name."""
    config = TableRecallConfig(
        fields=[TableRecallFieldConfig(input_column="description", indexed_column="description",
                                        minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
        max_results=max_results, min_total_match_value=0, include_field_scores=True
    )
    r = index.match(queries=pd.DataFrame({"description": [description]}), config=config)
    if len(r) == 0 or r["index_row"].iloc[0] == -1:
        return {"found": False, "matches": []}
    matches = [{
        "product_name": catalog.loc[catalog["description"] == row["description_candidate"], "product_name"].iloc[0],
        "match_confidence": int(row["description_score"]),
    } for _, row in r.iterrows()]
    return {"found": True, "matches": matches}


@server.tool()
def get_product_by_id(product_id: str) -> dict:
    """Look up the exact product record for a known product id. No fuzziness, an exact id in, the full record out."""
    config = TableRecallConfig(
        fields=[TableRecallFieldConfig(input_column="product_id", indexed_column="product_id",
                                        minimum_quality=0, weight=100, mode=TableRecallMode.EXACT)],
        max_results=1, min_total_match_value=0, include_field_scores=True
    )
    r = index.match(queries=pd.DataFrame({"product_id": [product_id]}), config=config)
    if len(r) == 0 or r["index_row"].iloc[0] == -1:
        return {"found": False}
    row = catalog.iloc[int(r["index_row"].iloc[0])]
    return {
        "found": True,
        "product_id": row["product_id"],
        "product_name": row["product_name"],
        "description": row["description"],
        "unit_price": float(row["unit_price"]),
    }


if __name__ == "__main__":
    server.run()


Overwriting mcp_server_advanced.py


## 3. Connect like a real client would

Same connection pattern as `03`: launch the server as a subprocess, speak JSON-RPC over stdin/stdout.

> **Known issue, being fixed upstream:** part of M|BOX's startup banner can slip past the stdout redirect in `mcp_server_advanced.py` on some platforms and land on the stdio transport. The MCP client tolerates this, it skips the malformed line and logs a warning rather than failing the session, but that warning is noisy enough to be distracting in a notebook. The cell below quiets that specific logger as a workaround; M|BOX itself is being changed so this redirect isn't necessary at all in an upcoming release.

In [5]:
import logging

# Workaround for a known M|BOX issue (fix planned upstream): a fragment of its
# startup banner can bypass the stdout redirect in mcp_server_advanced.py and
# reach the MCP transport. The client already skips the resulting malformed
# line safely, this just quiets the warning it logs when that happens.
logging.getLogger("mcp.client.stdio").setLevel(logging.CRITICAL)

In [6]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="python", args=["mcp_server_advanced.py"])

async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        print(f"{len(tools.tools)} tools, all served from the same loaded index:\n")
        for tool in tools.tools:
            print(f"- {tool.name}: {tool.description}")

4 tools, all served from the same loaded index:

- search_products: Search the product catalog by name, tolerating typos, optionally constrained by a maximum price.
- search_products_multi_field: Search by product name and, optionally, a rough description of what the product does.
Both fields are weighted and scored independently. Use this when the user describes what
they're looking for as much as, or instead of, naming it precisely.
- search_by_description: Search the product catalog by what a product does, for when the user doesn't remember its name.
- get_product_by_id: Look up the exact product record for a known product id. No fuzziness, an exact id in, the full record out.


Four tools, one process, one loaded index.

## 4. Call all four

The first reuses `03`'s worked example. The second is the one worth slowing down for: a query where *both* `product_name` and `description` are rough, and both meaningfully move the score, not just one field carrying the whole match.

In [7]:
import json

async def call(session, name, arguments):
    result = await session.call_tool(name, arguments)
    return json.loads(result.content[0].text)

async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        print("search_products, a typo'd name within budget:")
        print(json.dumps(await call(session, "search_products",
                                     {"product_name": "extendd batery pak", "max_price": 30}), indent=2))

search_products, a typo'd name within budget:
{
  "found": true,
  "matches": [
    {
      "product_name": "Extended Battery Pack",
      "match_confidence": 60,
      "overall_score": 72
    }
  ]
}


In [8]:
async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        print("search_products_multi_field, name AND description both rough:")
        print(json.dumps(await call(session, "search_products_multi_field",
                                     {"product_name": "battery pack", "description": "power cell for outdoor"}), indent=2))

search_products_multi_field, name AND description both rough:
{
  "found": true,
  "matches": [
    {
      "product_name": "Extended Battery Pack",
      "overall_score": 90,
      "field_scores": {
        "product_name": 92,
        "description": 88
      }
    }
  ]
}


`field_scores` shows `product_name` at `92` and `description` at `88`, both genuinely high, both pulling the `overall_score` of `90` up together. Neither field is a passenger here, the description alone would have been enough to find this product, and so would the name alone, this is what it looks like when they agree.

In [9]:
async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        print("search_by_description, no name given at all:")
        print(json.dumps(await call(session, "search_by_description",
                                     {"description": "night vision security camera"}), indent=2))

search_by_description, no name given at all:
{
  "found": true,
  "matches": [
    {
      "product_name": "Motion Sensor Camera",
      "match_confidence": 87
    }
  ]
}


In [10]:
async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        print("get_product_by_id, an exact id already in hand:")
        print(json.dumps(await call(session, "get_product_by_id", {"product_id": "B88-EXT"}), indent=2))
        print()
        print("get_product_by_id, an id that doesn't exist:")
        print(json.dumps(await call(session, "get_product_by_id", {"product_id": "ZZZ-999"}), indent=2))

get_product_by_id, an exact id already in hand:
{
  "found": true,
  "product_id": "B88-EXT",
  "product_name": "Extended Battery Pack",
  "description": "Rechargeable power cell for outdoor gear",
  "unit_price": 24.99
}

get_product_by_id, an id that doesn't exist:
{
  "found": false
}


Four distinct views, four distinct shapes of "what does the caller already know", fuzzy name, fuzzy name plus fuzzy description, description only, or an exact id with nothing fuzzy about it at all, and all four ran against the single index loaded once at server startup.

## 5. Practical notes

**Build the index once, load it many times.** Rebuilding from a CSV on every server start works fine for a four-row demo catalog and stops working fine well before you reach a real one. `to_binary()` once, `load_binary()` on every subsequent startup, is the same pattern as saving and loading indexes in `01-getting-started/`, it just happens to matter more once "restart" means "a server redeploying," not "a notebook kernel restarting."

**One index, many views, not one index per tool.** All four tools in this notebook call `.match()` against the exact same `index` object, loaded once, with a different `TableRecallConfig` per tool. Building a second index would have cost more memory and more load time for no benefit, the recall config, not the index, is what changes between a name search, a description search, and an exact id lookup.

**Rebuild the binary when the source data changes, not on a schedule.** `product_catalog_index.zip` is a snapshot. If the underlying catalog changes and the server keeps loading a stale `.zip`, it will confidently return stale answers, there's no staleness check built in. Rebuild and redeploy the binary as part of whatever process updates the source table.

**Keep noisy dependencies out of stdout.** Anything a library prints to stdout while an MCP stdio server is running is a potential protocol corruption, not just visual noise. M|BOX's own startup banner is the example in this notebook, redirecting stdout around the import catches most of it, and an upcoming M|BOX release removes the need for that redirect entirely, but until then, the client's tolerance for an occasional malformed line, skip it and log a warning rather than fail the session, is what kept this notebook's tool calls working end to end regardless.

## Next steps

- **`05-grounding_rag_with_deterministic_matching.ipynb`** - apply this same grounding principle to retrieval-augmented generation
- **`06-confidence_based_escalation.ipynb`** - use the match score itself to decide whether an agent should proceed, ask, or hand off to a human